# TPC-H Orders & Customers

**Dataset:** `samples.tpch.orders`, `samples.tpch.customer`

**Difficulty:** Medium

**Topics:** join, groupBy, window, subquery patterns

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window as W

orders = spark.read.table("samples.tpch.orders")
customer = spark.read.table("samples.tpch.customer")

## Problem 1

Join orders with customers on `o_custkey = c_custkey`. Compute total spend and order count per customer market segment. Sort by `total_spend` descending.

**Expected output columns:**
- `c_mktsegment`
- `total_spend`
- `order_count`

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1

result_1 = orders.join(customer, orders.o_custkey == customer.c_custkey).groupBy("c_mktsegment").agg(
    F.sum("o_totalPrice").alias("total_spend"),
    F.count("*").alias("order_count")
)

result_1.display()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'total_spend' in cols, "Missing column: total_spend"
assert 'order_count' in cols, "Missing column: order_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_spend = result_1.agg(F.min('total_spend')).collect()[0][0]
assert float(min_spend) >= 0, f"Expected total_spend >= 0, found min={min_spend}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Find the top 5 customers by total order value. Join orders and customers.

**Expected output columns:**
- `c_custkey`
- `c_name`
- `c_mktsegment`
- `total_order_value`

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2
order_agg = orders.groupBy("o_custkey").agg(
    F.sum("o_totalPrice").alias("total_order_value")
).orderBy(F.col("total_order_value").desc()).limit(5)
result_2 = customer.join(F.broadcast(order_agg), customer.c_custkey == order_agg.o_custkey).select(
    "c_custkey", "c_name", "c_mktsegment", "total_order_value"
)

result_2.display()

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'c_custkey' in cols, "Missing column: c_custkey"
assert 'c_name' in cols, "Missing column: c_name"
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'total_order_value' in cols, "Missing column: total_order_value"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 5, f"Expected at most 5 rows (top 5), got {cnt}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Compute average order value per market segment, including min and max order value.

**Expected output columns:**
- `c_mktsegment`
- `avg_order_value`
- `min_order_value`
- `max_order_value`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = orders.join(F.broadcast(customer), orders.o_custkey == customer.c_custkey).groupBy("c_mktsegment").agg(
    F.avg("o_totalPrice").alias("avg_order_value"),
    F.min("o_totalPrice").alias("min_order_value"),
    F.max("o_totalPrice").alias("max_order_value")
)

result_3.display()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'avg_order_value' in cols, "Missing column: avg_order_value"
assert 'min_order_value' in cols, "Missing column: min_order_value"
assert 'max_order_value' in cols, "Missing column: max_order_value"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Find customers who have placed orders under more than one order priority level.

**Expected output columns:**
- `c_custkey`
- `c_name`
- `priorities_used`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4
priority = orders.groupBy("o_custkey").agg(
    F.countDistinct("o_orderpriority").alias("priorities_used")
).filter(F.col("priorities_used") > 1)

result_4 = customer.join(F.broadcast(priority), customer.c_custkey == priority.o_custkey).select(
    "c_custkey",
    "c_name",
    "priorities_used"
)

result_4.display()

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'c_custkey' in cols, "Missing column: c_custkey"
assert 'c_name' in cols, "Missing column: c_name"
assert 'priorities_used' in cols, "Missing column: priorities_used"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_pri = result_4.agg(F.min('priorities_used')).collect()[0][0]
assert min_pri > 1, f"Expected priorities_used > 1 for all rows, found min={min_pri}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Using a window function, rank customers by total spend within each market segment. Keep only rank <= 3.

**Expected output columns:**
- `c_mktsegment`
- `c_name`
- `total_spend`
- `rank_in_segment`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5
order_agg = (
    orders
    .groupBy("o_custkey")
    .agg(F.sum("o_totalPrice").alias("total_spend"))
)

w = W.partitionBy("c_mktsegment").orderBy(F.col("total_spend").desc())
result_5 = (
    customer
    .join(F.broadcast(order_agg), customer["c_custkey"] == order_agg["o_custkey"])
    .select(
        "c_mktsegment",
        "c_name",
        "total_spend",
        F.rank().over(w).alias("rank_in_segment")
    )
    .filter(F.col("rank_in_segment") <= 3)
)

result_5.display()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'c_name' in cols, "Missing column: c_name"
assert 'total_spend' in cols, "Missing column: total_spend"
assert 'rank_in_segment' in cols, "Missing column: rank_in_segment"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rank = result_5.agg(F.max('rank_in_segment')).collect()[0][0]
assert max_rank <= 3, f"Expected rank_in_segment <= 3, found max={max_rank}"
min_rank = result_5.agg(F.min('rank_in_segment')).collect()[0][0]
assert min_rank >= 1, f"Expected rank_in_segment >= 1, found min={min_rank}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Count and compute revenue for high-priority orders (`1-URGENT` or `2-HIGH`) per market segment.

**Expected output columns:**
- `c_mktsegment`
- `high_priority_orders`
- `high_priority_revenue`

In [0]:
orders.limit(5).display()

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6

result_6 = (
    orders
    .filter(F.col("o_orderpriority").isin("1-URGENT","2-HIGH"))
    .join(F.broadcast(customer), orders["o_custkey"] == customer["c_custkey"])
    .groupBy("c_mktsegment")
    .agg(
        F.count("*").alias("high_priority_orders"),
        F.sum("o_totalPrice").alias("high_priority_revenue")
    )
)

result_6.display()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'high_priority_orders' in cols, "Missing column: high_priority_orders"
assert 'high_priority_revenue' in cols, "Missing column: high_priority_revenue"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_rev = result_6.agg(F.min('high_priority_revenue')).collect()[0][0]
assert float(min_rev) >= 0, f"Expected high_priority_revenue >= 0, found min={min_rev}"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Find customers who have never placed an order using a left anti-join.

**Expected output columns:**
- `c_custkey`
- `c_name`
- `c_mktsegment`
- `c_acctbal`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7

result_7 = (
    customer
    .join(orders, customer["c_custkey"] == orders["o_custkey"], "left_anti")
    .select("c_custkey", "c_name", "c_mktsegment", "c_acctbal")
)

result_7.display()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'c_custkey' in cols, "Missing column: c_custkey"
assert 'c_name' in cols, "Missing column: c_name"
assert 'c_mktsegment' in cols, "Missing column: c_mktsegment"
assert 'c_acctbal' in cols, "Missing column: c_acctbal"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
print(f"Problem 7 passed ✓  ({cnt} rows)")